<a href="https://colab.research.google.com/github/simplyshree/SeqTrainer/blob/issue-3-all-model-baselines/notebooks/final_training/ipromp_final_training_t4_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# iPro-MP final five-fold inference and stacker on a Colab T4

This notebook evaluates the official E. coli iPro-MP model on the exact shared
SeqTrainer splits. iPro-MP is inference-only: it has no epochs. The notebook
loads five official fold checkpoints sequentially, caches each split/fold
logit file, and compares ensemble methods using validation MCC only.

## Fixed contract

- GSE144621 EP_DNA_BERT2_genomic_order
- exact shared train, validation, and test CSV files
- labels 0 = non-promoter and 1 = promoter
- seed 42
- official E. coli species ID 10
- DNABERT-6 backbone and overlapping 6-mer tokenization
- maximum sequence token length 300
- five official fold checkpoints, loaded sequentially
- validation MCC, then validation AUPRC, selects the ensemble and threshold
- held-out test is evaluated once after selection

In [ ]:
# 1. Verify the requested accelerator.
import subprocess

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
print(gpu_info)
if "T4" not in gpu_info:
    raise RuntimeError("Select an NVIDIA T4 runtime, then rerun this notebook from the top.")

In [ ]:
# 2. Runtime configuration. Change only DRIVE_DATA_DIR if your Drive layout differs.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-all-model-baselines"
REPO_DIR = Path("/content/SeqTrainer")
MINIFORGE_DIR = Path("/content/miniforge3")
ENV_DIR = Path("/content/envs/seqtrainer-final-ipromp-t4")
ENV_PYTHON = ENV_DIR / "bin" / "python"

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Data")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/SeqTrainer/final_training/ipromp_seed42")
DRIVE_MODEL_CACHE = Path("/content/drive/MyDrive/SeqTrainer/final_training/model_cache/ipromp")
LOCAL_DATA_DIR = Path("/content/seqtrainer_final_training/data")
LOCAL_OUTPUT_DIR = Path("/content/seqtrainer_final_training/ipromp_seed42")
LOCAL_MODEL_ROOT = Path("/content/seqtrainer_final_training/ipromp_models")

RESUME_MODE = "latest"
RUN_TRAIN_SPLIT = True
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive data:", DRIVE_DATA_DIR)
print("Drive output:", DRIVE_OUTPUT_DIR)

In [ ]:
# 3. Create the pinned Python 3.10 environment.
installer = Path("/content/Miniforge3-Linux-x86_64.sh")
if not (MINIFORGE_DIR / "bin" / "conda").exists():
    subprocess.run(
        ["wget", "-q", "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh", "-O", str(installer)],
        check=True,
    )
    subprocess.run(["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)], check=True)
conda = MINIFORGE_DIR / "bin" / "conda"
if not ENV_PYTHON.exists():
    subprocess.run([str(conda), "create", "-y", "-p", str(ENV_DIR), "python=3.10", "pip"], check=True)
print("Environment:", ENV_PYTHON)

In [ ]:
# 4. Check out the requested branch without touching main or annotation-mvp.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
print("Branch:", subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 5. Install the pinned T4 inference environment.
torch_index = "https://download.pytorch.org/whl/cu121"
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools<70", "wheel"], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "torch==2.2.2", "--index-url", torch_index], check=True)
subprocess.run([
    str(ENV_PYTHON), "-m", "pip", "install",
    "transformers==4.29.2", "numpy==1.24.4", "pandas==2.0.3",
    "scikit-learn==1.3.2", "einops==0.6.1", "rdflib", "remotezip",
    "joblib", "matplotlib",
], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
print("Pinned packages installed.")

In [ ]:
# 6. Mount Drive and restore the model cache to local disk.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_MODEL_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
if any(DRIVE_MODEL_CACHE.iterdir()):
    shutil.copytree(DRIVE_MODEL_CACHE, LOCAL_MODEL_ROOT, dirs_exist_ok=True)
    print("Restored iPro-MP and DNABERT-6 model cache from Drive.")
else:
    print("No Drive model cache yet; the next cell downloads the official files.")

In [ ]:
# 7. Download missing official model files once, then persist the cache.
ipromp_dir = LOCAL_MODEL_ROOT / "ipromp_ecoli"
dnabert6_dir = LOCAL_MODEL_ROOT / "DNABERT-6"
folds = [ipromp_dir / f"10_fold_{fold}.pth" for fold in range(1, 6)]
dnabert6_files = [dnabert6_dir / name for name in ("config.json", "pytorch_model.bin", "vocab.txt")]

if not all(path.is_file() for path in folds):
    downloader = REPO_DIR / "notebooks/benchmarks_sg/ipromp_benchmark/download_ecoli_weights.py"
    ipromp_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([str(ENV_PYTHON), str(downloader), "--output-dir", str(ipromp_dir)], check=True)

if not all(path.is_file() for path in dnabert6_files):
    dnabert6_dir.mkdir(parents=True, exist_ok=True)
    download_code = (
        "from huggingface_hub import snapshot_download; "
        f"snapshot_download(repo_id='zhihan1996/DNA_bert_6', local_dir={str(dnabert6_dir)!r}, "
        "allow_patterns=['config.json','pytorch_model.bin','special_tokens_map.json','tokenizer_config.json','vocab.txt'])"
    )
    subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "huggingface_hub"], check=True)
    subprocess.run([str(ENV_PYTHON), "-c", download_code], check=True)

shutil.copytree(LOCAL_MODEL_ROOT, DRIVE_MODEL_CACHE, dirs_exist_ok=True)
print("Model cache ready:", LOCAL_MODEL_ROOT)

In [ ]:
# 8. Verify the isolated package and GPU before inference.
check = r'''
import json, sys, torch, transformers, seqtrainer
print(json.dumps({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seqtrainer": seqtrainer.__file__,
}, indent=2))
assert torch.cuda.is_available()
assert "T4" in torch.cuda.get_device_name(0)
'''
subprocess.run([str(ENV_PYTHON), "-c", check], check=True)

## 9. Run sequential fold inference and the train-fitted ensemble

The runner audits and stages the shared data, saves every fold/split logits
file with its input SHA-256, restores valid cached work after a disconnect,
fits the stacker on train only, selects the ensemble and threshold on
validation only, and then reports the selected ensemble on test.

In [ ]:
helper = REPO_DIR / "notebooks/final_training/helpers/run_ipromp_final.py"
run_env = os.environ.copy()
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

command = [
    str(ENV_PYTHON), str(helper),
    "--repo-dir", str(REPO_DIR),
    "--drive-data-dir", str(DRIVE_DATA_DIR),
    "--local-data-dir", str(LOCAL_DATA_DIR),
    "--drive-output-dir", str(DRIVE_OUTPUT_DIR),
    "--local-output-dir", str(LOCAL_OUTPUT_DIR),
    "--drive-model-cache", str(DRIVE_MODEL_CACHE),
    "--local-model-root", str(LOCAL_MODEL_ROOT),
    "--resume", RESUME_MODE,
]
if RUN_TRAIN_SPLIT:
    command.append("--run-train-split")
print("Running:", " ".join(command))
subprocess.run(command, check=True, env=run_env)

In [ ]:
# 10. Inspect the validation-selected ensemble and final held-out metrics.
import json
import pandas as pd

metrics = pd.read_csv(DRIVE_OUTPUT_DIR / "metrics.csv")
candidates = pd.read_csv(DRIVE_OUTPUT_DIR / "ensemble_candidates.csv")
selected = json.loads((DRIVE_OUTPUT_DIR / "selected_ensemble.json").read_text())
display(candidates)
display(metrics)
print("Selected ensemble:", selected["strategy"])
print("Validation MCC:", selected["validation_mcc"])
print("Validation AUPRC:", selected["validation_auprc"])
print("Held-out test MCC:", float(metrics.loc[metrics.split == "test", "mcc"].iloc[0]))
print("Held-out test AUPRC:", float(metrics.loc[metrics.split == "test", "auprc"].iloc[0]))

In [ ]:
# 11. Verify fold-level caches and artifact contract.
required = [
    "config.json", "input_split_audit.json", "environment.json", "history.csv",
    "metrics.csv", "metrics.json", "predictions.csv", "manifest.json",
    "ipromp_id_mapping.csv", "ensemble_candidates.csv", "selected_ensemble.json",
    "fold_predictions/train_fold_predictions.csv",
    "fold_predictions/validation_fold_predictions.csv",
    "fold_predictions/test_fold_predictions.csv",
    "plots/ensemble_validation_mcc.png",
]
missing = [name for name in required if not (DRIVE_OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError("Missing final iPro-MP artifacts: " + ", ".join(missing))
print("All iPro-MP artifacts are present under:", DRIVE_OUTPUT_DIR)
for path in sorted(DRIVE_OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DRIVE_OUTPUT_DIR), path.stat().st_size, "bytes")

## Final step: verify outputs and disconnect runtime

Do not place executable cells after this cell.

In [ ]:
import os
import time

required_outputs = [
    DRIVE_OUTPUT_DIR / "metrics.csv",
    DRIVE_OUTPUT_DIR / "metrics.json",
    DRIVE_OUTPUT_DIR / "predictions.csv",
    DRIVE_OUTPUT_DIR / "manifest.json",
    DRIVE_OUTPUT_DIR / "history.csv",
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError("Runtime will not disconnect because outputs are missing: " + ", ".join(missing))
print("All required artifacts were saved:")
for path in required_outputs:
    print("-", path)
if hasattr(os, "sync"):
    os.sync()
print("Disconnecting the Colab runtime in five seconds.")
time.sleep(5)
from google.colab import runtime
runtime.unassign()